# 07 — Z5 PyNite-only support and member-end release sensitivity

This notebook develops notebook 05 into a **PyNite-only** three-case workflow with editable support and connection assumptions.

The repository defaults intentionally reproduce the notebook-05 PyNite model:

- `SECTION_MODE = "isotropic"`;
- `CROSSING_DIAGONAL_MODE = "continuous"`;
- nodes **7, 8, 9, 10, 35, 36** are fully fixed;
- all other active nodes are free;
- frame members have **no end releases**;
- CSV truss members retain the adapter's automatic local `Ry/Rz` releases at both ends.

Editable inputs:

- `data/Z5_07_supports.csv` — node restraints;
- `data/Z5_07_frame_end_releases.csv` — frame-member end releases.

Releases are treated as **member-end properties, not node-wide properties**. A continuous parent member can therefore remain continuous through a panel point while a different physical member meeting at the same node is released.

## Configuration semantics

### Support CSV

`support_type` accepts:

- `free` — all 6 nodal DOFs free;
- `fixed` — all 6 nodal DOFs restrained;
- `pinned` — translations restrained and rotations free;
- `custom` — use the six `custom_*` Boolean columns.

### Frame-end release CSV

Each frame element has independent `i_end_type` and `j_end_type` values:

- `rigid` — no release;
- `bending_pin` — release local `Ry` and `Rz`, while retaining local torsion `Rx`;
- `custom` — use the corresponding six custom Boolean columns.

The custom fields map directly to PyNite's local member-end release flags. Translational releases are exposed because PyNite supports them, but they can readily create mechanisms and should be introduced deliberately.

Truss members are not listed in the frame-release CSV; their existing pin-ended PyNite approximation remains automatic.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "solver.py").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate the keratia project root.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results" / "z5_07_pynite_release_sensitivity"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from solver import read_model
from helpers import plot_signed_stress_3d, plot_deformed_shape
from pynite_adapter import (
    PyNiteRunOptions,
    build_pynite_model,
    pynite_displacements_dataframe,
    pynite_reactions_dataframe,
    pynite_member_forces_dataframe,
    summarize_pynite_result,
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)

## 1. Inputs

The load cases are copied unchanged from notebook 05. Edit the two CSV files to change support or frame-end assumptions.

In [ ]:
SECTION_MODE = "isotropic"
PRINCIPAL_ORIENTATION = "strong_inward"
CROSSING_DIAGONAL_MODE = "continuous"
DEFORMED_SHAPE_SCALE = 100.0

SUPPORT_CONFIG_CSV = DATA_DIR / "Z5_07_supports.csv"
FRAME_RELEASE_CONFIG_CSV = DATA_DIR / "Z5_07_frame_end_releases.csv"

Z5_CASES = {
    "Case_1": [
        {"node_label": "high_xneg", "Fx": 16250.0, "Fy": 8320.0, "Fz": -20000.0, "Mx": 0.0, "My": 0.0, "Mz": 0.0},
        {"node_label": "high_xpos", "Fx": 16250.0, "Fy": 8320.0, "Fz": -20000.0, "Mx": 0.0, "My": 0.0, "Mz": 0.0},
    ],
    "Case_2": [
        {"node_label": "high_xneg", "Fx": 48900.0, "Fy": 0.0, "Fz": -13400.0, "Mx": 0.0, "My": 0.0, "Mz": 0.0},
        {"node_label": "high_xpos", "Fx": 48900.0, "Fy": 0.0, "Fz": -13400.0, "Mx": 0.0, "My": 0.0, "Mz": 0.0},
    ],
    "Case_3": [
        {"node_label": "high_xneg", "Fx": 21600.0, "Fy": 38000.0, "Fz": -13400.0, "Mx": 0.0, "My": 0.0, "Mz": 0.0},
        {"node_label": "high_xpos", "Fx": 21600.0, "Fy": 38000.0, "Fz": -13400.0, "Mx": 0.0, "My": 0.0, "Mz": 0.0},
    ],
}

display(pd.DataFrame([
    {"case": case_name, **row}
    for case_name, rows in Z5_CASES.items()
    for row in rows
]))

In [ ]:
z5_model = read_model(
    "Z5",
    DATA_DIR / "Z5_nodes_aligned.csv",
    DATA_DIR / "Z5_elements_with_sections.csv",
)

support_config = pd.read_csv(SUPPORT_CONFIG_CSV)
release_config = pd.read_csv(FRAME_RELEASE_CONFIG_CSV)

print("Input Z5 nodes:", len(z5_model.nodes))
print("Input Z5 elements:", len(z5_model.elements))
print("Support config rows:", len(support_config))
print("Frame release config rows:", len(release_config))
display(support_config)
display(release_config)

## 2. Configuration parser and validation

The parser is intentionally strict: duplicate IDs, unknown presets, missing active nodes, or release rows that do not correspond to frame elements fail before analysis.

In [ ]:
def _as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if pd.isna(value):
        return False
    text = str(value).strip().lower()
    if text in {"true", "1", "yes", "y"}:
        return True
    if text in {"false", "0", "no", "n", ""}:
        return False
    raise ValueError(f"Cannot interpret {value!r} as Boolean")

def support_flags_from_row(row):
    preset = str(row["support_type"]).strip().lower()
    if preset == "free":
        return (False, False, False, False, False, False)
    if preset == "fixed":
        return (True, True, True, True, True, True)
    if preset == "pinned":
        return (True, True, True, False, False, False)
    if preset == "custom":
        return tuple(_as_bool(row[f"custom_{dof}"]) for dof in ["DX", "DY", "DZ", "RX", "RY", "RZ"])
    raise ValueError(f"Unknown support_type {row['support_type']!r} for node {row['node_id']}")

def release_flags_for_end(row, end):
    preset = str(row[f"{end}_end_type"]).strip().lower()
    if preset == "rigid":
        return (False, False, False, False, False, False)
    if preset == "bending_pin":
        return (False, False, False, False, True, True)
    if preset == "custom":
        prefix = f"custom_{end}_"
        return tuple(_as_bool(row[prefix + dof]) for dof in ["Dx", "Dy", "Dz", "Rx", "Ry", "Rz"])
    raise ValueError(f"Unknown {end}_end_type {row[f'{end}_end_type']!r} for element {row['element_id']}")

def validate_configs(working_model, support_cfg, release_cfg):
    if support_cfg["node_id"].duplicated().any():
        raise ValueError("Support CSV contains duplicate node_id rows")
    if release_cfg["element_id"].duplicated().any():
        raise ValueError("Frame release CSV contains duplicate element_id rows")

    active_nodes = set(working_model.nodes["node_id"].astype(int))
    configured_nodes = set(support_cfg["node_id"].astype(int))
    if active_nodes != configured_nodes:
        raise ValueError(
            "Support CSV does not match the continuous-crossing working model. "
            f"Missing={sorted(active_nodes-configured_nodes)}, extra={sorted(configured_nodes-active_nodes)}"
        )

    frame_ids = set(
        working_model.elements.loc[
            working_model.elements["element_type"].str.lower() == "frame", "element_id"
        ].astype(int)
    )
    configured_frame_ids = set(release_cfg["element_id"].astype(int))
    if frame_ids != configured_frame_ids:
        raise ValueError(
            "Frame release CSV does not match the working-model frame elements. "
            f"Missing={sorted(frame_ids-configured_frame_ids)}, extra={sorted(configured_frame_ids-frame_ids)}"
        )

def apply_support_config(fe, support_cfg):
    applied = []
    for _, row in support_cfg.sort_values("node_id").iterrows():
        flags = support_flags_from_row(row)
        nid = int(row["node_id"])
        fe.def_support(
            f"N{nid}",
            support_DX=flags[0], support_DY=flags[1], support_DZ=flags[2],
            support_RX=flags[3], support_RY=flags[4], support_RZ=flags[5],
        )
        applied.append({
            "node_id": nid, "node_label": row["node_label"], "support_type": row["support_type"],
            "DX": flags[0], "DY": flags[1], "DZ": flags[2],
            "RX": flags[3], "RY": flags[4], "RZ": flags[5],
        })
    return pd.DataFrame(applied)

def apply_frame_release_config(fe, release_cfg):
    applied = []
    for _, row in release_cfg.sort_values("element_id").iterrows():
        i = release_flags_for_end(row, "i")
        j = release_flags_for_end(row, "j")
        eid = int(row["element_id"])
        fe.def_releases(
            f"E{eid}",
            Dxi=i[0], Dyi=i[1], Dzi=i[2], Rxi=i[3], Ryi=i[4], Rzi=i[5],
            Dxj=j[0], Dyj=j[1], Dzj=j[2], Rxj=j[3], Ryj=j[4], Rzj=j[5],
        )
        applied.append({
            "element_id": eid, "start_node": int(row["start_node"]), "end_node": int(row["end_node"]),
            "i_end_type": row["i_end_type"], "j_end_type": row["j_end_type"],
            "i_Dx": i[0], "i_Dy": i[1], "i_Dz": i[2], "i_Rx": i[3], "i_Ry": i[4], "i_Rz": i[5],
            "j_Dx": j[0], "j_Dy": j[1], "j_Dz": j[2], "j_Rx": j[3], "j_Ry": j[4], "j_Rz": j[5],
        })
    return pd.DataFrame(applied)

def baseline_05_status(support_cfg, release_cfg):
    expected_fixed = {7, 8, 9, 10, 35, 36}
    all_supports_match = all(
        support_flags_from_row(row) ==
        ((True, True, True, True, True, True) if int(row.node_id) in expected_fixed
         else (False, False, False, False, False, False))
        for _, row in support_cfg.iterrows()
    )
    all_frames_rigid = all(
        release_flags_for_end(row, "i") == (False,) * 6
        and release_flags_for_end(row, "j") == (False,) * 6
        for _, row in release_cfg.iterrows()
    )
    actual_fixed = sorted(
        int(row.node_id) for _, row in support_cfg.iterrows()
        if support_flags_from_row(row) == (True, True, True, True, True, True)
    )
    return {
        "expected_fixed_nodes": sorted(expected_fixed),
        "actual_fully_fixed_nodes": actual_fixed,
        "supports_match_05": bool(all_supports_match),
        "frame_releases_match_05": bool(all_frames_rigid),
        "overall_matches_05_joint_support_inputs": bool(all_supports_match and all_frames_rigid),
    }

## 3. Baseline fingerprint

With the committed CSVs, `overall_matches_05_joint_support_inputs` should be `True`. After an intentional sensitivity edit it will normally become `False`; that is informational rather than an error.

In [ ]:
baseline_status = baseline_05_status(support_config, release_config)
display(pd.DataFrame([baseline_status]))

## 4. PyNite-only three-case solve

For this notebook, the adapter's automatic support inference is disabled so that the support CSV is the single source of truth. The committed CSV explicitly recreates notebook 05's six fixed supports. Truss end releases remain automatic; frame-end releases are then applied from the second CSV.

In [ ]:
pynite_options = PyNiteRunOptions(
    crossing_diagonal_mode=CROSSING_DIAGONAL_MODE,
    section_mode=SECTION_MODE,
    principal_orientation=PRINCIPAL_ORIENTATION,
    fix_min_z_nodes=False,
    fix_nodes_marked_as_support=False,
    release_truss_end_bending=True,
    auto_fix_truss_only_rotations=True,
)

def solve_configured_pynite_case(case_name, loads):
    fe, working_model, section_table, crossing_diag, loads_used = build_pynite_model(
        z5_model, loads, options=pynite_options, case_name=case_name, combo_name=case_name
    )
    validate_configs(working_model, support_config, release_config)
    supports_applied = apply_support_config(fe, support_config)
    frame_releases_applied = apply_frame_release_config(fe, release_config)

    fe.analyze_linear(log=False, check_stability=True, check_statics=False, sparse=True)

    displacements = pynite_displacements_dataframe(fe, working_model, case_name)
    reactions = pynite_reactions_dataframe(fe, working_model, case_name)
    member_forces = pynite_member_forces_dataframe(
        fe, working_model, section_table, pynite_options, case_name
    )

    validation = pd.DataFrame([{
        "model": "Z5", "backend": "PyNite", "case": case_name,
        "section_mode": SECTION_MODE,
        "crossing_diagonal_mode": CROSSING_DIAGONAL_MODE,
        "support_config": SUPPORT_CONFIG_CSV.name,
        "frame_release_config": FRAME_RELEASE_CONFIG_CSV.name,
        "matches_05_joint_support_inputs": baseline_status["overall_matches_05_joint_support_inputs"],
    }])

    return {
        "name": "Z5", "backend": "PyNite", "pynite_model": fe,
        "nodes": working_model.nodes.copy(), "elements": working_model.elements.copy(),
        "displacements": displacements, "reactions": reactions, "member_forces": member_forces,
        "section_properties": section_table, "loads_used": loads_used,
        "validation": validation, "crossing_mode_diagnostics": crossing_diag,
        "supports_applied": supports_applied,
        "frame_releases_applied": frame_releases_applied,
        "options": pynite_options,
    }

z5_pynite_results = {}
for case_name, loads in Z5_CASES.items():
    print(f"Solving Z5 {case_name} with configured PyNite model...")
    z5_pynite_results[case_name] = solve_configured_pynite_case(case_name, loads)

pynite_summary = pd.DataFrame([
    {"case": case_name, **summarize_pynite_result(result)}
    for case_name, result in z5_pynite_results.items()
])
display(pynite_summary)

## 5. Three-case governing comparison

As in notebook 05, an element is listed only when Case 1 exceeds **both** Case 2 and Case 3 in at least one physically meaningful demand measure.

In [ ]:
def stacked_member_results(case_results):
    frames = []
    for case_name, result in case_results.items():
        table = result["member_forces"].copy()
        table.insert(0, "case", case_name)
        frames.append(table)
    return pd.concat(frames, ignore_index=True).sort_values(["case", "element_id"]).reset_index(drop=True)

def _strictly_greater_than_both(c1, c2, c3, rtol=1e-7, atol=1e-9):
    other = max(float(c2), float(c3))
    c1 = float(c1)
    return (c1 > other) and (not np.isclose(c1, other, rtol=rtol, atol=atol))

def case1_governing_elements(case_results):
    indexed = {
        case: result["member_forces"].set_index("element_id")
        for case, result in case_results.items()
    }
    ids = sorted(set.intersection(*(set(df.index) for df in indexed.values())))
    rows = []
    for eid in ids:
        rs = [indexed[c].loc[eid] for c in ["Case_1", "Case_2", "Case_3"]]
        def demands(r):
            N = float(r["axial_force_signed_N"])
            return {
                "abs_axial_kN": float(r["max_abs_N_N"]) / 1000.0,
                "compression_kN": max(-N, 0.0) / 1000.0,
                "tension_kN": max(N, 0.0) / 1000.0,
                "sigma_abs_MPa": float(r["sigma_max_abs_MPa"]),
                "bending_kNmm": float(r["max_M_resultant_Nmm"]) / 1000.0,
            }
        d1, d2, d3 = map(demands, rs)
        metrics = [m for m in d1 if _strictly_greater_than_both(d1[m], d2[m], d3[m])]
        if not metrics:
            continue
        r1 = rs[0]
        rows.append({
            "element_id": int(eid),
            "element_type": r1.get("element_type", ""),
            "start_label": r1.get("start_label", ""),
            "end_label": r1.get("end_label", ""),
            "member_role": r1.get("member_role", ""),
            "section_name": r1.get("section_name", ""),
            "governing_metrics": ", ".join(metrics),
            "C1_axial_signed_kN": float(rs[0]["axial_force_signed_N"]) / 1000.0,
            "C2_axial_signed_kN": float(rs[1]["axial_force_signed_N"]) / 1000.0,
            "C3_axial_signed_kN": float(rs[2]["axial_force_signed_N"]) / 1000.0,
            "C1_sigma_abs_MPa": d1["sigma_abs_MPa"],
            "C2_sigma_abs_MPa": d2["sigma_abs_MPa"],
            "C3_sigma_abs_MPa": d3["sigma_abs_MPa"],
            "C1_bending_kNmm": d1["bending_kNmm"],
            "C2_bending_kNmm": d2["bending_kNmm"],
            "C3_bending_kNmm": d3["bending_kNmm"],
        })
    return pd.DataFrame(rows).sort_values("element_id").reset_index(drop=True) if rows else pd.DataFrame()

pynite_case1_governing = case1_governing_elements(z5_pynite_results)
pynite_all_element_results = stacked_member_results(z5_pynite_results)

print("Elements where Case 1 governs over BOTH Case 2 and Case 3:", len(pynite_case1_governing))
display(pynite_case1_governing)
display(pynite_all_element_results)

## 6. Applied-support and release audit

These tables show the interpreted configuration actually sent to PyNite.

In [ ]:
sample = z5_pynite_results["Case_1"]
display(sample["supports_applied"])

active_releases = sample["frame_releases_applied"].copy()
flag_cols = [
    c for c in active_releases.columns
    if (c.startswith("i_") or c.startswith("j_"))
    and c not in {"i_end_type", "j_end_type"}
]
active_releases = active_releases[active_releases[flag_cols].any(axis=1)]
print("Frame elements with at least one active release:", len(active_releases))
display(active_releases)

## 7. Export result tables

In [ ]:
pynite_summary.to_csv(RESULTS_DIR / "pynite_summary.csv", index=False)
pynite_all_element_results.to_csv(RESULTS_DIR / "pynite_all_element_results.csv", index=False)
pynite_case1_governing.to_csv(RESULTS_DIR / "pynite_case1_governs_both.csv", index=False)
sample["supports_applied"].to_csv(RESULTS_DIR / "supports_applied.csv", index=False)
sample["frame_releases_applied"].to_csv(RESULTS_DIR / "frame_releases_applied.csv", index=False)
print("Wrote CSVs to:", RESULTS_DIR)

## 8. Axial tension/compression visualization

In [ ]:
for case_name, result in z5_pynite_results.items():
    fig, ax = plot_signed_stress_3d(
        result["nodes"], result["elements"], result["member_forces"],
        title=f"Z5 PyNite axial tension/compression — {case_name}",
        stress_column="sigma_axial_signed_MPa",
        annotate_load_points=True, annotate_elements=False,
        show_frame=True, show_truss=True,
    )
    plt.show()

## 9. Deformed-shape visualization

In [ ]:
for case_name, result in z5_pynite_results.items():
    fig, ax = plot_deformed_shape(
        result["nodes"], result["elements"], result["displacements"],
        scale=DEFORMED_SHAPE_SCALE,
        title=f"Z5 PyNite deformed shape — {case_name}",
        show_truss=True,
    )
    plt.show()

## 10. Suggested first sensitivity sequence

The committed version is deliberately **R0 / notebook-05-equivalent**. Suggested later experiments are not pre-applied:

1. free nodes 35 and 36 in `Z5_07_supports.csv`;
2. restore the baseline and introduce selected `bending_pin` releases at physical frame-member ends at nodes 0–6;
3. only after those results are understood, investigate base joints 7–10 separately.

Do not apply a node-wide release merely because several members meet there. A split segment of one continuous physical parent member can remain rigid through the node while a different parent member at the same node is released.